# Export human heat stress
- This script is used to export metrics of human heat stress.
- Simulations: CNTL, TranAlbe

In [2]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
home_path = '/gws/ssde/j25a/duicv/yuansun/'
project_path = f'{home_path}0_wrf-cstm_GM-HK/'
var_list = ['HUMIDEX_U', 'SWBGT_U', 'HIA_U']

In [ ]:
ds_mask_hk = xr.open_dataset(f'{project_path}HK/mask/mask_HK_lat_lon.nc')
ds_mask_hk_pct_urban = ds_mask_hk['PCT_URBAN'] # values: 0-1
region='HK'
tmax_time_zone = 6 # daily maximum

In [5]:
for scenario in ['cntl', 'tran_albe']:
    wrfout_path = f'{project_path}runs/TranUrbAlb_{region}/runs/d04_{scenario}/archive_ctsm/lnd/'
    wrfout_path_files = os.listdir(wrfout_path)
    nc_files = [f for f in wrfout_path_files if f.endswith('.nc')  and '.clm2.h0.' in f]
    total_results = []
    for files in nc_files:
        ds = xr.open_dataset(f'{wrfout_path}{files}')
        var_max_average_list = []
        for var in var_list: 
            ds_var = ds[var].where(ds_mask_hk['mask'] == 1)
            ds_var = ds_var.assign_coords(time=ds_var.time.dt.round("h"))
            ds_var_max = ds_var.sel(time=ds_var.time.dt.hour == tmax_time_zone)   
            ds_var_max_average = (ds_var_max * ds_mask_hk_pct_urban).sum(dim=['lat', 'lon'], skipna=True) / ds_mask_hk_pct_urban.sum(dim=['lat', 'lon'], skipna=True)
            df_var_max_average = ds_var_max_average.to_dataframe(var).reset_index()
            var_max_average_list.append(df_var_max_average)
        df_var_max_average = pd.merge(var_max_average_list[0], var_max_average_list[1], on='time', how='inner').merge(var_max_average_list[2], on='time', how='inner')
        total_results.append(df_var_max_average)
    df_var_max_average = pd.concat(total_results, axis=0, ignore_index=True)
    output_filename = f'./data_for_figure/{region}_human_heat_stress_{scenario}.csv'
    df_var_max_average['time'] = pd.to_datetime(df_var_max_average['time'])
    df_var_max_clean = df_var_max_average.sort_values(by='time')
    df_var_max_clean.to_csv(output_filename, index=False)
    df_var_max_clean